# Human Activity Recognition — Clustering Analysis

Unsupervised clustering of smartphone accelerometer/gyroscope sensor data to uncover hidden human-activity groupings without labels.

## 1. Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

df = pd.read_csv("human_activity_master.csv")
print(df.shape)
df.head()

## 2. Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

## 3. Preprocessing

In [ ]:
X = df.copy()
print(X.shape)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

### Remove highly correlated features

In [ ]:
corr = X_scaled.corr().abs()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]

X_clean = X_scaled.drop(columns=to_drop)

print("Dropped:", len(to_drop))
print("Remaining features:", X_clean.shape[1])

The dataset contains 561 sensor-derived features from smartphone accelerometer and gyroscope signals. Because many features are highly redundant, we removed features with correlation higher than 0.95. This reduces multicollinearity and improves clustering performance by preventing biased distance calculations. After standardization, all features were scaled to have zero mean and unit variance, ensuring that no feature dominates due to scale differences.

## 4. Choosing the Number of Clusters

### Elbow Method

In [ ]:
inertia = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_clean)
    inertia.append(kmeans.inertia_)

plt.plot(K_range, inertia, marker='o')
plt.title("Elbow Method")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.show()

### Silhouette Analysis

In [ ]:
sil_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_clean)
    sil_scores.append(silhouette_score(X_clean, labels))

plt.plot(K_range, sil_scores, marker='o')
plt.title("Silhouette Score")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Score")
plt.show()

The optimal number of clusters was determined using both the Elbow Method and Silhouette Analysis. The elbow curve shows a diminishing improvement after a certain number of clusters, indicating that adding more clusters does not significantly reduce inertia. Additionally, the silhouette score was used to evaluate cluster separation quality, where higher values indicate better-defined clusters. Based on both methods, we selected k = 4 as the best trade-off between compactness and separation.

## 5. KMeans Clustering

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_clean)

In [ ]:
df_clusters = X_clean.copy()
df_clusters["Cluster"] = clusters

df_clusters.head()

### PCA Visualization

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_clean)

plt.scatter(X_pca[:,0], X_pca[:,1], c=clusters, cmap='viridis', s=10)
plt.title("KMeans Clusters (PCA Projection)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()

### Cluster Centers

In [ ]:
centers = pd.DataFrame(kmeans.cluster_centers_, columns=X_clean.columns)
centers

## 6. Anomaly Detection

In [ ]:
distances = np.min(kmeans.transform(X_clean), axis=1)

In [ ]:
threshold = np.percentile(distances, 95)
anomalies = X_clean[distances >= threshold]

print(anomalies.shape)

## 7. Comparing Clustering Algorithms

In [ ]:
agglo = AgglomerativeClustering(n_clusters=4)
agglo_labels = agglo.fit_predict(X_clean)

agglo_silhouette = silhouette_score(X_clean, agglo_labels)
print("Agglomerative Silhouette:", agglo_silhouette)

In [ ]:
gmm = GaussianMixture(n_components=4, random_state=42)
gmm_labels = gmm.fit_predict(X_clean)

gmm_silhouette = silhouette_score(X_clean, gmm_labels)
print("GMM Silhouette:", gmm_silhouette)

In [ ]:
birch = Birch(n_clusters=4)
birch_labels = birch.fit_predict(X_clean)

birch_silhouette = silhouette_score(X_clean, birch_labels)
print("BIRCH Silhouette:", birch_silhouette)

In [ ]:
results = pd.DataFrame({
    "Model": ["KMeans", "Agglomerative", "GMM", "BIRCH"],
    "Silhouette Score": [
        silhouette_score(X_clean, clusters),
        agglo_silhouette,
        gmm_silhouette,
        birch_silhouette
    ]
})

results

### Comparison of Clustering Models

The performance of the clustering algorithms was evaluated using the Silhouette Score, which measures how well-separated and compact the clusters are.

## 📌 Final Conclusion

This project demonstrated how unsupervised learning can uncover hidden structures in human motion data without using labels.

Key findings include:

- Feature scaling and correlation removal were essential for meaningful clustering
- KMeans successfully identified general motion categories
- PCA visualization confirmed partial separation of clusters
- Anomaly detection revealed unusual or transitional movements
- GMM provided the best statistical clustering performance due to its probabilistic nature